# Intelligent SQL Assistant with Automatic Query Correction
A self-refining natural language → SQL pipeline that uses schema context and execution feedback
to iteratively generate and improve SQL queries.

In [1]:
import os
import json
import re
import pandas as pd
import utils
import pyodbc
import aisuite as ai
from dotenv import load_dotenv

load_dotenv()

client = ai.Client()

### GET Sql Schema

In [2]:
driver = os.getenv("DATABASE_DRIVER")
server = os.getenv("DATABASE_SERVER")
database = os.getenv("DATABASE_NAME")

schema = utils.get_schema_with_relations(driver=driver, server=server, database=database)

print(schema)


TABLE: __EFMigrationsHistory
  MigrationId nvarchar
  ProductVersion nvarchar

TABLE: CartItems
  UserId int
  ProductId int
  ProductTypeId int
  Title nvarchar
  ProductTypeName nvarchar
  ImageUrl nvarchar
  Price decimal
  Qantity int
  FOREIGN KEY (ProductId) REFERENCES Products(Id)
  FOREIGN KEY (ProductTypeId) REFERENCES ProductTypes(Id)
  FOREIGN KEY (UserId) REFERENCES Users(Id)

TABLE: Categories
  Id int
  Name nvarchar
  Url nvarchar

TABLE: Products
  Id int
  Title nvarchar
  Description nvarchar
  ImageUrl nvarchar
  CategoryId int
  Featured bit
  FOREIGN KEY (CategoryId) REFERENCES Categories(Id)

TABLE: ProductTypes
  Id int
  Name nvarchar

TABLE: ProductVariants
  ProductId int
  ProductTypeId int
  Price decimal
  OriginalPrice decimal
  FOREIGN KEY (ProductId) REFERENCES Products(Id)
  FOREIGN KEY (ProductTypeId) REFERENCES ProductTypes(Id)

TABLE: Users
  Id int
  Email nvarchar
  PasswordHash varbinary
  PasswordSalt varbinary
  DateCreated datetime2


### Ask LLM to explain the schema

In [16]:
def explain_schema(schema: str, model: str = "openai:gpt-4.1") -> str:
    """
    Use an LLM to provide:
    1) Table explanations
    2) Foreign key relationships
    3) Suggested queries for the database
    4) Suggested reporting queries / analytics queries

    Returns a dictionary with keys:
        - 'table_explanations': {table_name: explanation}
        - 'relationships': list of relationship strings
        - 'optimization suggestions': list of optimization suggestions
        - 'suggested_queries': list of suggested queries
        - 'reporting_queries': list of reporting/analytics query ideas
    """

    prompt = f"""
You are an expert database analyst specializing in Microsoft SQL Server. Given the following SQL database schema, provide the following:

1) A brief explanation in plain english of each table, describing its purpose, the meaning of its main columns, and what type of data it stores.

2) A list of foreign key relationships between tables in plain english (e.g., "orders.customer_id references customers.id").

3) Provide any missing optimizations for the schema (such as indexing, normalization, naming consistency, or adding foreign key constraints).

   For each recommended optimization:
   - Assume SQL Server as the target database.
   - Determine whether the object (index, constraint, etc.) already exists.
   - If it does not exist, provide a CREATE statement.
   - If it exists but differs, provide a safe conditional DROP followed by CREATE.
   - Use SQL Server-safe conditional checks (sys.indexes, sys.foreign_keys, etc.).

4) Suggest 5 useful SQL queries someone might want to run on this database, each with a short description and the SQL.

5) Suggest 3-5 reporting or analytics queries suitable for management dashboards (trends, aggregates, top-N metrics, etc.), each with a short description and the SQL.

Schema:
{schema}

Return STRICT JSON with this exact structure and property order (do not add or omit fields):

{{
  "table_explanations": {{
    "<table_name>": "<explanation>"
  }},
  "relationships": [
    "<table1.column -> table2.column>"
  ],
  "optimisation_suggestions": [
    "<description and SQL Server-safe optimization statements>"
  ],
  "suggested_queries": [
    "<description and SQL>"
  ],
  "reporting_queries": [
    "<description and SQL>"
  ]
}}
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You must return only the JSON object described in the prompt. Do not include explanations, reasoning, or commentary outside the JSON. Do not return empty lists unless the schema truly contains no information."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=3500
    )

    content = response.choices[0].message.content.strip()

    # Strip any accidental code block markers
    content = re.sub(r"^```(?:json)?|```$", "", content, flags=re.MULTILINE).strip()

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        print("Failed to parse JSON. Raw output:")
        print(content)
        # Fallback
        result = {
            "table_explanations": {},
            "relationships": [],
            "suggested_queries": [],
            "reporting_queries": [],
            "optimisation_suggestions": []
        }

    return result


In [17]:
schema_info = explain_schema(schema)

In [18]:
print(json.dumps(schema_info, indent=2))

{
  "table_explanations": {
    "__EFMigrationsHistory": "Stores the history of Entity Framework migrations applied to the database. 'MigrationId' uniquely identifies each migration, and 'ProductVersion' records the version of Entity Framework used.",
    "CartItems": "Represents items currently in users' shopping carts. 'UserId' links to the user, 'ProductId' and 'ProductTypeId' specify the product and its type, 'Title', 'ProductTypeName', and 'ImageUrl' provide product details, 'Price' is the price per item, and 'Qantity' (likely a typo for 'Quantity') is the number of units.",
    "Categories": "Defines product categories. 'Id' is the unique identifier, 'Name' is the category name, and 'Url' is a URL-friendly string for the category.",
    "Products": "Stores product information. 'Id' is the unique identifier, 'Title' and 'Description' describe the product, 'ImageUrl' is the product image, 'CategoryId' links to the category, and 'Featured' indicates if the product is featured.",
   

In [19]:
print("Table Explanations:")
for table, explanation in schema_info['table_explanations'].items():
    print(f"{table}: {explanation}\n")

print("Relationships:")
for rel in schema_info['relationships']:
    print(f"- {rel}")

print("Suggested Queries:")
for q in schema_info['suggested_queries']:
    parts = q.split(":")
    print(f"{parts[0]}")
    print(f"{parts[1]}")

print("Reporting Queries:")
for q in schema_info['reporting_queries']:
    parts = q.split(":")
    print(f"{parts[0]}")
    print(f"{parts[1]}")

print("Optimisation Suggestions:")
for s in schema_info['optimisation_suggestions']:
    print(f"- {s}")

Table Explanations:
__EFMigrationsHistory: Stores the history of Entity Framework migrations applied to the database. 'MigrationId' uniquely identifies each migration, and 'ProductVersion' records the version of Entity Framework used.

CartItems: Represents items currently in users' shopping carts. 'UserId' links to the user, 'ProductId' and 'ProductTypeId' specify the product and its type, 'Title', 'ProductTypeName', and 'ImageUrl' provide product details, 'Price' is the price per item, and 'Qantity' (likely a typo for 'Quantity') is the number of units.

Categories: Defines product categories. 'Id' is the unique identifier, 'Name' is the category name, and 'Url' is a URL-friendly string for the category.

Products: Stores product information. 'Id' is the unique identifier, 'Title' and 'Description' describe the product, 'ImageUrl' is the product image, 'CategoryId' links to the category, and 'Featured' indicates if the product is featured.

ProductTypes: Lists the types or variations

In [7]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for MS SQLSERVER.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    sql_query = response.choices[0].message.content.strip()
    return sql_query

In [8]:
# We ask a question about the data in natural language
question = "I need to display list of products on web page, provide sql query, take into consideration products, Categories, ProductTypes and ProductVariants?"

utils.print_html(question, title="User Question")
sql_query_v1 = generate_sql(question, schema, model="openai:gpt-4.1")
utils.print_html(sql_query_v1, title="Generated SQL Query")